In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import os
import time

In [2]:
headers = {
    'authority': 'www.99acres.com',
    'accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7',
    'accept-language': 'en-US,en;q=0.9',
    'cache-control': 'no-cache',
    'dnt': '1',
    'pragma': 'no-cache',
    'referer': 'https://www.99acres.com/residential-land-in-gurgaon-ffid-page',
    'sec-ch-ua': '"Chromium";v="107", "Not;A=Brand";v="8"',
    'sec-ch-ua-mobile': '?0',
    'sec-ch-ua-platform': '"macOS"',
    'sec-fetch-dest': 'document',
    'sec-fetch-mode': 'navigate',
    'sec-fetch-site': 'same-origin',
    'sec-fetch-user': '?1',
    'upgrade-insecure-requests': '1',
    'user-agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/527.36 (KHTML, like Gecko) Chrome/107.0.0.0 Safari/537.36',
}

In [3]:
land = pd.DataFrame()

In [9]:
import time

from bs4 import BeautifulSoup
from urllib.parse import urljoin

# Configuration
start = 1
end = 50

project_dir = r"C:\Users\THARUN TEJA S\ML_PROJECT"
City = "Gurgaon"

# CSV path
csv_file = os.path.join(
    project_dir,
    "Data",
    City,
    "Residential",
    f"residential_land_gurgaon_data-p{start}-{end}.csv"
)

# Create directory if it doesn't exist
os.makedirs(os.path.dirname(csv_file), exist_ok=True)

# Initialize
pageNumber = start
req = 0
land = pd.DataFrame()

session = requests.Session()
session.headers.update(headers)

# Helper function to safely extract text
def get_text(soup, selector):
    element = soup.select_one(selector)
    return element.get_text(strip=True) if element else ''

# Scraping
while pageNumber <= end:

    i = 0
    page_records = []

    url = (
        f'https://www.99acres.com/'
        f'residential-land-in-{City.lower()}-ffid-page-{pageNumber}'
    )

    print(f"\nProcessing page: {pageNumber}")

    try:
        response = session.get(url, timeout=30)
        response.raise_for_status()

        pageSoup = BeautifulSoup(response.content, 'html.parser')
        req += 1

    except requests.RequestException as e:
        print(f"Request failed on page {pageNumber}: {e}")
        pageNumber += 1
        continue

    # Find search container
    search_container = pageSoup.select_one(
        'div[data-label="SEARCH"]'
    )

    if search_container is None:
        print("Search container not found.")
        print("Status code:", response.status_code)
        print("Final URL:", response.url)

        pageNumber += 1
        continue

    properties = search_container.select(
        'section[data-hydration-on-demand="true"]'
    )

    if not properties:
        print(f"No property listings on page {pageNumber}")

        pageNumber += 1
        continue

    # Extract individual properties
    for soup in properties:

        property_name = get_text(
            soup, 'a.srpTuple__propertyName'
        )

        property_link = soup.select_one(
            'a.srpTuple__propertyName'
        )

        if not property_name or not property_link:
            continue

        link = urljoin(url, property_link.get('href', ''))

        society = get_text(
            soup, '#srp_tuple_society_heading'
        )

        area = get_text(
            soup, '#srp_tuple_secondary_area'
        )

        # Property detail page
        try:
            detail_response = session.get(
                link, timeout=30
            )
            detail_response.raise_for_status()

            dpageSoup = BeautifulSoup(
                detail_response.content, 'html.parser'
            )

            req += 1

        except requests.RequestException as e:
            print(f"Detail page failed: {link} | {e}")
            continue

        # Extract details
        price = get_text(dpageSoup, '#pdPrice2')

        address = get_text(dpageSoup, '#address')

        floorNumAllowed = get_text(
            dpageSoup, '#floorNumLabel'
        )

        noOfOpenSides = get_text(
            dpageSoup, '#noOfOpenSides'
        )

        possession = get_text(
            dpageSoup, '#agePossessionLbl'
        )

        # Nearby locations
        nearby_container = dpageSoup.select_one(
            'div.NearByLocation__tagWrap'
        )

        nearbyLocations = (
            [x.get_text(strip=True) for x in
             nearby_container.select('span.NearByLocation__infoText')]
            if nearby_container else []
        )

        # Description
        description = get_text(
            dpageSoup, '#description'
        )

        # Features
        feature_sections = dpageSoup.select('#features')

        features = (
            [x.get_text(strip=True) for x in
             feature_sections[0].select('li')]
            if feature_sections else []
        )

        # Ratings
        rating_container = dpageSoup.select_one(
            'div.review__rightSide>div>ul>li>div'
        )

        rating = (
            [x.get_text(strip=True) for x in
             rating_container.select(
                 'div.ratingByFeature__circleWrap'
             )]
            if rating_container else []
        )

        # Property ID
        property_id = get_text(
            dpageSoup, '#Prop_Id'
        )

        # Store property data
        property_data = {
            'property_name': property_name,
            'link': link,
            'society': society,
            'price': price,
            'areaWithType': area,
            'address': address,
            'floorNumAllowed': floorNumAllowed,
            'noOfOpenSides': noOfOpenSides,
            'possession': possession,
            'nearbyLocations': nearbyLocations,
            'description': description,
            'features': features,
            'rating': rating,
            'property_id': property_id
        }

        page_records.append(property_data)
        i += 1

        # Respect website request limits
        if req % 4 == 0:
            time.sleep(10)

        if req % 20 == 0:
            time.sleep(20)

    # Save current page data
    if page_records:

        temp_df = pd.DataFrame(page_records)

        land = pd.concat(
            [land, temp_df],
            ignore_index=True
        )

        file_exists = os.path.isfile(csv_file)

        temp_df.to_csv(
            csv_file,
            mode='a',
            header=not file_exists,
            index=False
        )

    print(f"{pageNumber} -> {i} properties extracted")

    pageNumber += 1

    # Pause between listing pages
    time.sleep(3)

print("\nScraping completed!")

print("Total properties extracted:", len(land))
print("CSV saved at:", csv_file)



Processing page: 1
Request failed on page 1: 403 Client Error: Forbidden for url: https://www.99acres.com/residential-land-in-gurgaon-ffid-page-1

Processing page: 2
Request failed on page 2: 403 Client Error: Forbidden for url: https://www.99acres.com/residential-land-in-gurgaon-ffid-page-2

Processing page: 3
Request failed on page 3: 403 Client Error: Forbidden for url: https://www.99acres.com/residential-land-in-gurgaon-ffid-page-3

Processing page: 4
Request failed on page 4: 403 Client Error: Forbidden for url: https://www.99acres.com/residential-land-in-gurgaon-ffid-page-4

Processing page: 5
Request failed on page 5: 403 Client Error: Forbidden for url: https://www.99acres.com/residential-land-in-gurgaon-ffid-page-5

Processing page: 6
Request failed on page 6: 403 Client Error: Forbidden for url: https://www.99acres.com/residential-land-in-gurgaon-ffid-page-6

Processing page: 7
Request failed on page 7: 403 Client Error: Forbidden for url: https://www.99acres.com/residential

In [10]:

import os
import pandas as pd

def combine_csv_files(folder_path, combined_file_path):

    all_data = []

    for file_name in os.listdir(folder_path):

        if not file_name.endswith('.csv'):
            continue

        # Skip the combined output file
        if file_name == os.path.basename(combined_file_path):
            continue

        file_path = os.path.join(folder_path, file_name)

        if os.path.getsize(file_path) == 0:
            print(f"Skipping empty file: {file_name}")
            continue

        try:
            df = pd.read_csv(file_path)

            if df.empty:
                continue

            all_data.append(df)

            print(f"Read: {file_name} ({len(df)} rows)")

        except pd.errors.EmptyDataError:
            print(f"Empty CSV: {file_name}")

    if not all_data:
        print("No valid CSV files found.")
        return

    # Combine all CSV files
    combined_data = pd.concat(all_data, ignore_index=True)

    # Save combined dataset
    combined_data.to_csv(combined_file_path, index=False)

    print("\nCombined CSV saved successfully!")
    print("Total properties:", len(combined_data))
    print("Saved at:", combined_file_path)


# Windows paths
project_dir = r"C:\Users\THARUN TEJA S\ML_PROJECT"

folder_path = os.path.join(
    project_dir,
    "Data",
    "Gurgaon",
    "Residential"
)

combined_file_path = os.path.join(
    folder_path,
    "residential-land.csv"
)

combine_csv_files(folder_path, combined_file_path)


No valid CSV files found.
